In [1]:
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters
from lignin_saf.ligsaf_system import create_rcf_system
from lignin_saf.ligsaf_purification_system import create_rcf_oil_purification_system
from lignin_saf.monomer_purification import create_monomer_purification_system
from lignin_saf.ligsaf_utilities_system import create_rcf_utilities_system
from biorefineries import cellulosic
from cellulosic_tea import create_cellulosic_ethanol_tea
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo


In [2]:
# Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [3]:
chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d')



In [4]:
rcf_system = create_rcf_system(ins=poplar_in)
# rcf_system.simulate()

rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_Oil)
monomer_purification_sys = create_monomer_purification_system(ins=F.Purified_RCF_Oil)
# rcf_oil_purification_sys.simulate()
# monomer_purification_sys.simulate()
BT, WWT, gas_mixer = create_rcf_utilities_system()

etoh_system = cellulosic.create_cellulosic_ethanol_system(ins = F.Carbohydrate_Pulp)
#etoh_system.simulate()

rcf_combined_system = bst.System(
    'Combined_RCF_System',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, etoh_system, WWT),
    facilities=[gas_mixer, BT],
)
rcf_combined_system.simulate()
rcf_combined_system.show()



#integrated_tea = create_cellulosic_ethanol_tea(rcf_combined_system)

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: biogas> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: sludge> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: RO_treated_water> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: brine> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\process_tools\system_factory.py:279: RuntimeWarning: alias 'recycled_water' of <Stream: RO_treated_water> has been replaced in registry with <Stream: RO_treated_water>
  self.f(ins, outs, **kwargs)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermo

System: Combined_RCF_System
Highest convergence error among components in recycle
stream M604-0 after 1 loops:
- flow rate   5.03e+00 kmol/hr (0.017%)
- temperature 2.13e-06 K (6.9e-07%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] warm_process_water_1  
    phase: 'l', T: 368.15 K, P: 476228 Pa
    flow (kmol/hr): Water  870
[2] pretreatment_steam  
    phase: 'g', T: 541.15 K, P: 1.31722e+06 Pa
    flow (kmol/hr): Water  906
[3] warm_process_water_2  
    phase: 'l', T: 368.15 K, P: 476228 Pa
    flow (kmol/hr): Water  6.48e+03
[4] ammonia_process_water  
    phase: 'l', T: 368.15 K, P: 476228 Pa
    flow (kmol/hr): Water  3.31e+03
[5] sulfuric_acid  
    phase: 'l', T: 294.15 K, P: 547155 Pa
    flow (kmol/hr): Water  4.81
                    H2SO4  12.2
[6] cellulase  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water      788
                    Cellulase  31.1
[7] saccharification_water  
    phase: 'l', T

In [5]:
integrated_tea = create_cellulosic_ethanol_tea(rcf_combined_system)

In [6]:
print(f'The MSP for RCF monomers is  {round(integrated_tea.solve_price(F.RCF_Monomers),3)} USD/kg')

The MSP for RCF monomers is  25.465 USD/kg


In [7]:
rcf_combined_system

System: Combined_RCF_System
Highest convergence error among components in recycle
stream M604-0 after 1 loops:
- flow rate   5.03e+00 kmol/hr (0.017%)
- temperature 2.13e-06 K (6.9e-07%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] warm_process_water_1  
    phase: 'l', T: 368.15 K, P: 476228 Pa
    flow (kmol/hr): Water  870
[2] pretreatment_steam  
    phase: 'g', T: 541.15 K, P: 1.31722e+06 Pa
    flow (kmol/hr): Water  906
[3] warm_process_water_2  
    phase: 'l', T: 368.15 K, P: 476228 Pa
    flow (kmol/hr): Water  6.48e+03
[4] ammonia_process_water  
    phase: 'l', T: 368.15 K, P: 476228 Pa
    flow (kmol/hr): Water  3.31e+03
[5] sulfuric_acid  
    phase: 'l', T: 294.15 K, P: 547155 Pa
    flow (kmol/hr): Water  4.81
                    H2SO4  12.2
[6] cellulase  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water      788
                    Cellulase  31.1
[7] saccharification_water  
    phase: 'l', T

In [8]:
etoh_system.units[34]

MolecularSieve: U401
ins...
[0] s80  from  HXutility-H402
    phase: 'g', T: 388.15 K, P: 162120 Pa
    flow (kmol/hr): Water    130
                    Ethanol  545
                    NH3      0.751
outs...
[0] s81  to  Mixer-M402
    phase: 'g', T: 388.15 K, P: 162120 Pa
    flow (kmol/hr): Water    120
                    Ethanol  88.4
[1] s82  to  HXutility-H403
    phase: 'g', T: 388.15 K, P: 162120 Pa
    flow (kmol/hr): Water    9.72
                    Ethanol  457
                    NH3      0.751
